# Does agreement among peers predict stock price catch-up?

## 1. Hypothesis Pitch 

When a stock fails to move while its 1 year rolling related peers move together in one direction, it tends to catch up over the next five trading days. Specifically, I think the catch-up is stronger when the peer move reflects agreement across several peers than when the same average move comes from one exceptional peer. I measure agreement as the standard deviation of the five peers returns on the signal day. The lower it is, the more alike the peers move. I assume that T's five-day catch-up should be larger when that standard deviation is close to 0, and smaller when it is high. 

### Example

Both rows have the same +2% average peer return, and the standard deviation is 0% in scenario 1 and about 4.5% in scenario 2. I predict the target T rises, and it rises more where the standard deviation is 0%. 

| Metric | Scenario 1: Broad Agreement | Scenario 2: One Exceptional Peer |
|---|---:|---:|
| Peer A's return today | +2% | +10% |
| Peer B's return today | +2% | 0% |
| Peer C's return today | +2% | 0% |
| Peer D's return today | +2% | 0% |
| Peer E's return today | +2% | 0% |
| **Average of the 5 peers** | **+2%** | **+2%** |
| **Spread of the 5 peers (standard deviation)** | **0%** | **4.5%** |
| Target stock T's return today | +0.2% | +0.2% |
| **T's return over the next 5 days (my prediction)** | **Large catch-up, e.g. +1.4%** | **Small catch-up, e.g. +0.3%** |

### Why I think it might be true

When several related stocks rise by similar amounts, this may be because of information that benefits the whole group and reflect shared business condition. Investors may react quickly to some companies but take longer for others. This could leave stock T temporarily behind, allowing it to catch up. 

However, when the same average peer return comes mainly from one exceptional stock, the rise may reflect news specific to that company, such as its own earnings. There is then less reason for T to follow. This is why I expect stronger catch-up when the spread of peer returns is lower.


### What would prove me wrong

I would abandon the hypothesis if across a large sample, low standard deviation peer groups do not produce greater five day catch up than high standard deviation groups with similar average peer returns and similar initial low absolute return in T.

I would also abandon its use as a trading strategy if the catch up happens before I can enter at the next trading session’s open, or if transaction costs remove the benefit. If the results are too uncertain to draw a clear conclusion, I would report the evidence as inconclusive.



# Data Pipeline

### 1. Data Retrieval

In [45]:
import pandas as pd
import yfinance as yf


class MarketDataFetcher:
    """Fetch daily OHLCV market data for the tickers."""

    TICKERS = ["AAPL", "MSFT", "NVDA", "TSLA", "SPY"]

    def fetch_data(self, ticker: str, period: str | None = None) -> pd.DataFrame | None:
        """Fetch daily OHLCV data for one ticker."""
        try:
            data = yf.download(
                ticker,
                start="2008-12-01",
                end="2026-01-01",
                interval="1d",
                auto_adjust=True, 
                multi_level_index=False,
                progress=False,
                timeout=15,
            )
            if data.empty:
                print(f"No data found for {ticker}")
                return None
            
            if isinstance(data.columns, pd.MultiIndex):
                data.columns = data.columns.get_level_values(0)

            data.columns = [column.lower() for column in data.columns]
            required_columns = ["open", "high", "low", "close", "volume"]
            if not all(column in data.columns for column in required_columns):
                print(
                    f"Missing required columns in {ticker} data. "
                    f"Available: {list(data.columns)}"
                )
                return None

            print(f"Fetched {len(data)} daily bars for {ticker}")
            return data

        except Exception as error:
            print(f"Error fetching {ticker} data: {error}")
            return None

    def fetch_all_data(self, period: str | None = None) -> dict[str, pd.DataFrame]:
        """Fetch daily data for every configured ticker."""
        all_data = {}
        for ticker in self.TICKERS:
            data = self.fetch_data(ticker, period)
            if data is not None:
                all_data[ticker] = data
        return all_data


fetcher = MarketDataFetcher()
data = fetcher.fetch_all_data()


Fetched 4298 daily bars for AAPL
Fetched 4298 daily bars for MSFT
Fetched 4298 daily bars for NVDA
Fetched 3902 daily bars for TSLA
Fetched 4298 daily bars for SPY


In [46]:
import pandas as pd
import yfinance as yf
 
 
class MarketDataFetcher:
    """Fetch daily OHLCV market data for the tickers, in chunks."""
 
    TICKERS = ["AAPL", "MSFT", "NVDA", "TSLA"]
    CHUNK_SIZE = 100
 
    def fetch_chunk(self, tickers: list[str]) -> pd.DataFrame | None:
        """Fetch daily OHLCV data for one chunk of tickers."""
        try:
            data = yf.download(
                tickers,
                start="2008-12-01",
                end="2026-01-01",
                interval="1d",
                auto_adjust=True,
                progress=False,
                timeout=15,
            )
 
            if data.empty:
                print(f"No data found for chunk starting with {tickers[0]}")
                return None
 
            print(f"Fetched {len(data)} daily bars for {len(tickers)} tickers")
            return data
 
        except Exception as error:
            print(f"Error fetching chunk starting with {tickers[0]}: {error}")
            return None
 
    def fetch_all_data(self) -> pd.DataFrame | None:
        """Fetch daily data for every ticker. Returns one combined table with
        two-level columns (field, ticker), e.g. data["Close"]["AAPL"]."""
        chunks = []
        for i in range(0, len(self.TICKERS), self.CHUNK_SIZE):
            data = self.fetch_chunk(self.TICKERS[i : i + self.CHUNK_SIZE])
            if data is not None:
                chunks.append(data)
 
        if not chunks:
            return None
 
        combined = pd.concat(chunks, axis=1) # concat horizontally =1
 
        close = combined["Close"]
        failed = [t for t in self.TICKERS if t not in close.columns or close[t].isna().all()]
        print(f"{len(failed)} tickers failed: {failed}")
 
        return combined
 
 
fetcher = MarketDataFetcher()
data = fetcher.fetch_all_data()
 
if data is not None:  # the data/ folder must exist
    data.to_csv("data/raw_market_data.csv")
    data.to_parquet("data/raw_market_data.parquet")

Fetched 4298 daily bars for 4 tickers
0 tickers failed: []


#### Example of Saved Data: NVIDIA and MSFT

This is how the 'data' pd variable looks like once saved to csv (with rounded prices and dropped volume for readability)

| Price | Close | Close | High | High | Low | Low | Open | Open |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Ticker | MSFT | NVDA | MSFT | NVDA | MSFT | NVDA | MSFT | NVDA |
| Date | | | | | | | | |
| 12/1/08 | 13.51 | 0.16 | 14.48 | 0.17 | 13.50 | 0.16 | 14.43 | 0.17 |
| 12/2/08 | 13.90 | 0.17 | 14.02 | 0.17 | 13.47 | 0.16 | 13.78 | 0.16 |
| 12/3/08 | 14.42 | 0.18 | 14.44 | 0.18 | 13.50 | 0.16 | 13.54 | 0.16 |
| 12/4/08 | 13.87 | 0.17 | 14.46 | 0.18 | 13.64 | 0.16 | 14.08 | 0.17 |
| 12/5/08 | 14.42 | 0.17 | 14.50 | 0.17 | 13.41 | 0.16 | 13.73 | 0.16 |

### 2. Data Cleaning
1. Check for all missing nan values in the thing 
